# Phân loại mức độ ưu tiên phản ánh khách hàng

**Mô hình:** TF-IDF kết hợp Multinomial Naive Bayes  
**Các nhãn:** `khan_cap`, `binh_thuong`, `gop_y`

Notebook xây dựng quy trình hoàn chỉnh để đọc dữ liệu phản ánh tiếng Việt, làm sạch văn bản, huấn luyện mô hình, đánh giá, lưu mô hình và dự đoán phản ánh mới.

## 1. Cài đặt thư viện

Chạy ô dưới đây một lần nếu môi trường chưa có các thư viện cần thiết. Dấu `%` cho phép lệnh cài đặt chạy đúng trong Jupyter Notebook.

In [1]:
%pip install pandas scikit-learn matplotlib seaborn joblib ipywidgets -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import thư viện

Các thư viện được dùng để xử lý dữ liệu, biểu diễn văn bản bằng TF-IDF, huấn luyện Naive Bayes, đánh giá kết quả, trực quan hóa ma trận nhầm lẫn và lưu mô hình.

In [2]:
from pathlib import Path
import re
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

RANDOM_STATE = 42
TEN_NHAN = {
    "khan_cap": "Khẩn cấp",
    "binh_thuong": "Bình thường",
    "gop_y": "Góp ý",
}

print("Đã import thư viện thành công.")

Đã import thư viện thành công.


## 3. Đọc dữ liệu CSV

File CSV cần có hai cột:

- `noi_dung`: nội dung phản ánh của khách hàng.
- `nhan`: một trong ba giá trị `khan_cap`, `binh_thuong`, `gop_y`.

Notebook sẽ tìm file `phan_anh_khach_hang.csv` trong cùng thư mục với notebook hoặc trong thư mục `data`. Nếu chưa tìm thấy, hãy đặt file vào một trong hai vị trí rồi chạy lại ô.

In [3]:
cac_duong_dan = [
    Path("phan_anh_khach_hang.csv"),
    Path("data") / "phan_anh_khach_hang.csv",
]
duong_dan_csv = next((p for p in cac_duong_dan if p.exists()), None)

if duong_dan_csv is None:
    raise FileNotFoundError(
        "Không tìm thấy phan_anh_khach_hang.csv. "
        "Hãy đặt file cạnh notebook hoặc trong thư mục data/."
    )

du_lieu = pd.read_csv(duong_dan_csv, encoding="utf-8-sig")

cot_bat_buoc = {"noi_dung", "nhan"}
if not cot_bat_buoc.issubset(du_lieu.columns):
    raise ValueError("CSV phải có đúng hai cột bắt buộc: noi_dung và nhan.")

du_lieu = du_lieu.dropna(subset=["noi_dung", "nhan"]).copy()
du_lieu["noi_dung"] = du_lieu["noi_dung"].astype(str)
du_lieu["nhan"] = du_lieu["nhan"].astype(str).str.strip()
du_lieu = du_lieu.drop_duplicates(subset=["noi_dung"]).reset_index(drop=True)

nhan_hop_le = set(TEN_NHAN)
nhan_khong_hop_le = set(du_lieu["nhan"]) - nhan_hop_le
if nhan_khong_hop_le:
    raise ValueError(f"Phát hiện nhãn không hợp lệ: {sorted(nhan_khong_hop_le)}")

print(f"Đã đọc {len(du_lieu)} dòng từ: {duong_dan_csv.resolve()}")
display(du_lieu.head())
display(du_lieu["nhan"].value_counts().rename("so_luong").to_frame())

FileNotFoundError: Không tìm thấy phan_anh_khach_hang.csv. Hãy đặt file cạnh notebook hoặc trong thư mục data/.

## 4. Tiền xử lý văn bản

Hàm dưới đây chuyển văn bản về chữ thường, xóa đường dẫn, dấu câu, ký tự đặc biệt và khoảng trắng thừa. Các chữ cái có dấu tiếng Việt và những từ phủ định như **không**, **chưa** vẫn được giữ lại vì chúng có ý nghĩa phân loại.

In [ ]:
def tien_xu_ly_van_ban(van_ban):
    """Làm sạch một nội dung phản ánh tiếng Việt."""
    if not isinstance(van_ban, str):
        return ""
    van_ban = van_ban.lower()
    van_ban = re.sub(r"https?://\S+|www\.\S+", " ", van_ban)
    van_ban = re.sub(r"[^\w\s]", " ", van_ban, flags=re.UNICODE)
    van_ban = re.sub(r"_+", " ", van_ban)
    return re.sub(r"\s+", " ", van_ban).strip()

du_lieu["noi_dung_sach"] = du_lieu["noi_dung"].apply(tien_xu_ly_van_ban)
du_lieu = du_lieu[du_lieu["noi_dung_sach"].str.len() > 0].reset_index(drop=True)

display(du_lieu[["noi_dung", "noi_dung_sach", "nhan"]].head(10))

## 5. Chia tập huấn luyện và kiểm tra

Dữ liệu được chia theo tỷ lệ 80% huấn luyện và 20% kiểm tra. Tham số `stratify=y` giúp giữ tỷ lệ các nhãn gần giống nhau ở hai tập.

In [ ]:
X = du_lieu["noi_dung_sach"]
y = du_lieu["nhan"]

if y.nunique() < 2 or y.value_counts().min() < 2:
    raise ValueError("Mỗi nhãn cần có ít nhất 2 mẫu để chia train/test.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Số mẫu huấn luyện: {len(X_train)}")
print(f"Số mẫu kiểm tra:   {len(X_test)}")
display(pd.crosstab(index=y_train, columns="train"))
display(pd.crosstab(index=y_test, columns="test"))

## 6. Biểu diễn văn bản bằng TF-IDF

TF-IDF chuyển văn bản thành vector số dựa trên mức độ quan trọng của từ. `ngram_range=(1, 2)` cho phép mô hình học cả từ đơn như **tiền** và cụm hai từ như **trừ tiền**. Bộ TF-IDF chỉ được học trên tập train để tránh rò rỉ dữ liệu kiểm tra.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=1,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Kích thước ma trận train:", X_train_tfidf.shape)
print("Kích thước ma trận test: ", X_test_tfidf.shape)
print("Số đặc trưng TF-IDF:     ", len(tfidf.get_feature_names_out()))

## 7. Huấn luyện Multinomial Naive Bayes

Multinomial Naive Bayes là thuật toán phân loại xác suất phù hợp với dữ liệu văn bản. Mô hình học những từ và cụm từ thường đi cùng từng mức độ ưu tiên.

In [ ]:
mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train_tfidf, y_train)

y_du_doan = mo_hinh.predict(X_test_tfidf)
print("Huấn luyện mô hình hoàn tất.")

## 8. Đánh giá mô hình

- **Accuracy:** tỷ lệ dự đoán đúng trên toàn bộ tập kiểm tra.
- **Precision:** mức chính xác của các mẫu được dự đoán vào một lớp.
- **Recall:** khả năng tìm ra các mẫu thực sự thuộc lớp đó.
- **F1-score:** trung bình điều hòa giữa Precision và Recall.

Notebook hiển thị cả giá trị tổng hợp theo `macro average` và báo cáo chi tiết cho từng nhãn. Với bài toán này, Recall của lớp `khan_cap` đặc biệt quan trọng vì không nên bỏ sót phản ánh nghiêm trọng.

In [ ]:
accuracy = accuracy_score(y_test, y_du_doan)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_du_doan,
    average="macro",
    zero_division=0,
)

bang_chi_so = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"],
    "Giá trị": [accuracy, precision, recall, f1],
})
bang_chi_so["Giá trị"] = bang_chi_so["Giá trị"].round(4)
display(bang_chi_so)

thu_tu_nhan = [nhan for nhan in TEN_NHAN if nhan in set(y_test)]
bao_cao = classification_report(
    y_test,
    y_du_doan,
    labels=thu_tu_nhan,
    target_names=[TEN_NHAN[nhan] for nhan in thu_tu_nhan],
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(bao_cao).T.round(4))

### Ma trận nhầm lẫn

Hàng biểu diễn nhãn thực tế, cột biểu diễn nhãn mô hình dự đoán. Các số trên đường chéo là những trường hợp được dự đoán đúng.

In [ ]:
ma_tran = confusion_matrix(y_test, y_du_doan, labels=thu_tu_nhan)

plt.figure(figsize=(7, 5))
sns.heatmap(
    ma_tran,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[TEN_NHAN[nhan] for nhan in thu_tu_nhan],
    yticklabels=[TEN_NHAN[nhan] for nhan in thu_tu_nhan],
)
plt.title("Ma trận nhầm lẫn")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.tight_layout()
plt.savefig("ma_tran_nham_lan.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Lưu mô hình bằng joblib

Mô hình Naive Bayes và bộ TF-IDF được lưu vào thư mục `model`. Có thể tải lại hai file này để dự đoán mà không cần huấn luyện lại.

In [ ]:
thu_muc_model = Path("model")
thu_muc_model.mkdir(exist_ok=True)

joblib.dump(mo_hinh, thu_muc_model / "mo_hinh_naive_bayes.pkl")
joblib.dump(tfidf, thu_muc_model / "tfidf.pkl")

print("Đã lưu:")
print("-", (thu_muc_model / "mo_hinh_naive_bayes.pkl").resolve())
print("-", (thu_muc_model / "tfidf.pkl").resolve())

## 10. Dự đoán thử phản ánh mới

Hàm sau nhận một câu phản ánh, áp dụng đúng bước tiền xử lý và TF-IDF đã học, rồi trả về nhãn cùng xác suất của cả ba lớp.

In [ ]:
def du_doan_phan_anh(noi_dung):
    noi_dung_sach = tien_xu_ly_van_ban(noi_dung)
    if not noi_dung_sach:
        raise ValueError("Nội dung phản ánh không được để trống.")

    vector = tfidf.transform([noi_dung_sach])
    nhan_du_doan = mo_hinh.predict(vector)[0]
    xac_suat = mo_hinh.predict_proba(vector)[0]
    bang_xac_suat = pd.DataFrame({
        "Nhãn": [TEN_NHAN.get(nhan, nhan) for nhan in mo_hinh.classes_],
        "Xác suất": xac_suat,
    }).sort_values("Xác suất", ascending=False)
    return nhan_du_doan, bang_xac_suat

cau_thu = "Tôi bị trừ tiền hai lần nhưng đơn hàng chưa được tạo"
nhan, bang_xac_suat = du_doan_phan_anh(cau_thu)
print("Nội dung:", cau_thu)
print("Kết quả:", TEN_NHAN.get(nhan, nhan))
bang_hien_thi = bang_xac_suat.copy()
bang_hien_thi["Xác suất"] = bang_hien_thi["Xác suất"].map(lambda x: f"{x:.2%}")
display(bang_hien_thi)

### Ô nhập văn bản tương tác

Nhập phản ánh vào ô bên dưới và bấm **Phân loại**. Nếu Jupyter yêu cầu, hãy bật tiện ích `ipywidgets` hoặc khởi động lại kernel sau khi cài thư viện.

In [ ]:
import ipywidgets as widgets

o_nhap = widgets.Textarea(
    value="",
    placeholder="Ví dụ: Tài khoản của tôi bị người lạ đăng nhập",
    description="Phản ánh:",
    layout=widgets.Layout(width="80%", height="100px"),
    style={"description_width": "80px"},
)
nut_phan_loai = widgets.Button(description="Phân loại", button_style="primary")
vung_ket_qua = widgets.Output()

def khi_bam_nut(_):
    with vung_ket_qua:
        clear_output()
        try:
            nhan, bang_xac_suat = du_doan_phan_anh(o_nhap.value)
            print("Mức độ ưu tiên:", TEN_NHAN.get(nhan, nhan))
            bang_hien_thi = bang_xac_suat.copy()
            bang_hien_thi["Xác suất"] = bang_hien_thi["Xác suất"].map(lambda x: f"{x:.2%}")
            display(bang_hien_thi)
        except ValueError as loi:
            print("Lỗi:", loi)

nut_phan_loai.on_click(khi_bam_nut)
display(o_nhap, nut_phan_loai, vung_ket_qua)

## 11. Kết luận

Notebook đã hoàn thành toàn bộ quy trình phân loại phản ánh khách hàng bằng TF-IDF và Multinomial Naive Bayes. Mô hình có ưu điểm là đơn giản, huấn luyện nhanh, không cần GPU và dễ giải thích. Hạn chế chính là mô hình dựa nhiều vào từ xuất hiện trong dữ liệu nên chưa hiểu sâu ngữ cảnh, câu mơ hồ, lỗi chính tả hoặc phản ánh chứa nhiều ý.